In [ ]:
from ppktstore.registry import configure_phenopacket_registry
import pandas as pd
from ppkt2synergy import PhenotypeDatasetBuilder, HPOCorrelationAnalyzer

## Loading cohorts from phenopacket store

In [ ]:
def run_global_cohort_scan():

    registry = configure_phenopacket_registry()

    significant_cohorts_summary = []
    all_sig_tables = []

    with registry.open_phenopacket_store() as ps:
        available_cohorts = [c.name for c in ps.cohorts()]
        print(f"Total cohorts found: {len(available_cohorts)}")

        for i, cohort_name in enumerate(available_cohorts, 1):
            print(f"[{i}/{len(available_cohorts)}] Processing cohort: {cohort_name}")

            try:
                phenopackets = list(ps.iter_cohort_phenopackets(cohort_name))

                if len(phenopackets) < 20:
                    continue

                dataset = PhenotypeDatasetBuilder(phenopackets).build(
                    build_gpsea_cohort=False
                )

                analyzer = HPOCorrelationAnalyzer(dataset=dataset)
                results_table = analyzer.compute_correlation_matrix().results_table

                if results_table.empty:
                    continue

                sig_results = results_table[results_table["adj_p_value"] < 0.05]
                if len(sig_results)==0:
                    continue

                summary_row = {
                    "cohort": cohort_name,
                    "n_patients": len(phenopackets),
                    "tested_pairs": len(results_table),
                    "adj_signif_counts": len(sig_results),
                }

                significant_cohorts_summary.append(summary_row)

                if len(sig_results):
                    all_sig_tables.append(sig_results.assign(Cohort=cohort_name))

            except Exception:
                continue

    summary_df = pd.DataFrame(significant_cohorts_summary)

    master_sig_df = (
        pd.concat(all_sig_tables, ignore_index=True)
        if all_sig_tables else pd.DataFrame()
    )

    return summary_df, master_sig_df

In [9]:
summary_df, master_sig_df = run_global_cohort_scan()

Calculating pairwise correlation: 100%|██████████| 3/3 [00:00<00:00, 6275.77it/s]
[Correlation Analysis Blocked]: No HPO term pairs passed the candidate pre-filtering selection.
--------------------------------------------------------------------------------------------------
DIAGNOSIS SUMMARY:
  - Pairs remaining after HPO Hierarchy Masking (excluding ancestors/descendants): 5550
  - Pairs dropped due to low sample size (min_individuals_for_correlation_test=20): 5550
SUGGESTION:
  Try lowering `min_individuals_for_correlation_test` (e.g., to 10 or 5) when instantiating HPOCorrelationAnalyzer,
  or check the sample size and missing value distribution in your Phenopackets queue.
--------------------------------------------------------------------------------------------------
Calculating pairwise correlation: 100%|██████████| 623/623 [00:00<00:00, 4598.77it/s]


In [10]:
display(summary_df.sort_values("adj_signif_counts", ascending=False))

summary_df.to_csv("cohort_summary.csv", index=False)

if not master_sig_df.empty:
    display(master_sig_df.head(20))
    master_sig_df.to_csv("all_significant_pairs.csv", index=False)

,cohort,n_patients,tested_pairs,adj_signif_counts
19,FBN1,143,477,52
33,ITPR1,170,427,38
26,GLI3,81,82,20
20,FBXL4,95,1117,20
32,INPP4A,30,102,17
18,EPG5,39,38,11
5,ANKRD11,333,895,9
17,EHMT1,125,1588,7
27,GNAS,103,63,5
1,ACBD6,45,2520,4


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids,Cohort
0,HP:0000508,Ptosis,HP:0008551,Microtia,0.535656,9.865830e-06,0.008726,19,4,12,34,69,1,15266616,11q_terminal_deletion
1,HP:0000322,Short philtrum,HP:0000343,Long philtrum,-0.547376,6.490005e-06,0.008726,16,36,16,1,69,1,15266616,11q_terminal_deletion
2,HP:0000445,Wide nose,HP:0005280,Depressed nasal bridge,1.000000,7.989277e-10,0.000002,12,0,0,24,36,1,37951597,ACBD6
3,HP:0000280,Coarse facial features,HP:0000729,Autistic behavior,0.846154,3.164882e-05,0.025346,11,0,2,11,24,1,37951597,ACBD6
4,HP:0000280,Coarse facial features,HP:0001257,Spasticity,0.686607,4.023195e-05,0.025346,9,6,0,22,37,1,37951597,ACBD6
5,HP:0000280,Coarse facial features,HP:0002061,Lower limb spasticity,0.717137,3.816877e-05,0.025346,9,5,0,20,34,1,37951597,ACBD6
6,HP:0000403,Recurrent otitis media,HP:0000405,Conductive hearing impairment,0.755012,6.694051e-07,0.000206,27,2,3,13,45,1,36446582,ANKRD11
7,HP:0000405,Conductive hearing impairment,HP:0000407,Sensorineural hearing impairment,1.000000,4.709085e-07,0.000206,82,0,0,4,86,1,36446582,ANKRD11
8,HP:0001642,Pulmonic stenosis,HP:0001647,Bicuspid aortic valve,0.794048,6.903526e-07,0.000206,167,1,1,4,173,1,36446582,ANKRD11
9,HP:0000316,Hypertelorism,HP:0000411,Protruding ear,0.603683,6.199220e-06,0.001387,21,10,2,23,56,1,36446582,ANKRD11
